# Análises de Engenharia

In [0]:
#carregando setup basico
from pyspark.sql import functions as F
CATALOGO, SCHEMA = "workspace", "default"

daqui pra frente vou dar preferência ao SQL para responder as perguntas do projeto

## 1. Curva de potência
como a potência média se comporta em função do vento? Bate com o esperado (corte ~3 m/s, nominal ~2,05 MW) do manual? Alguma turbina fica sistematicamente abaixo das outras (subperformance)?

para potencia media:

In [0]:
%sql
SELECT
    floor(wind_speed_m_s * 2)/2 AS faixa_vento, --- agrupando de 0,5 em 0,5 m/s para facilitar a analise
    round(avg(power_kw),1) AS potencia_media_kw,
    count(*) AS n_leituras
FROM workspace.default.gold_fato_scada_curado
WHERE power_kw IS NOT NULL
AND wind_speed_m_s IS NOT NULL
GROUP BY floor(wind_speed_m_s * 2)/2
ORDER BY faixa_vento;

faixa_vento,potencia_media_kw,n_leituras
0.0,-1.0,297
0.5,-1.0,1544
1.0,-1.0,3125
1.5,-1.5,5826
2.0,-1.1,10592
2.5,8.3,15812
3.0,40.6,16631
3.5,85.9,18567
4.0,138.4,21254
4.5,202.6,23250


Databricks visualization. Run in Databricks to view.

para cada uma das 6 turbinas:

In [0]:
%sql
SELECT
    floor(wind_speed_m_s * 2)/2 AS faixa_vento, turbina_id,
    round(avg(power_kw),1) AS potencia_media_kw
FROM workspace.default.gold_fato_scada_curado
WHERE power_kw IS NOT NULL
AND wind_speed_m_s IS NOT NULL
GROUP BY floor(wind_speed_m_s * 2)/2, turbina_id
ORDER BY faixa_vento, turbina_id;


faixa_vento,turbina_id,potencia_media_kw
0.0,1,-0.5
0.0,2,-0.9
0.0,3,-1.6
0.0,4,-0.4
0.0,5,-1.1
0.0,6,-1.4
0.5,1,-0.5
0.5,2,-0.9
0.5,3,-1.4
0.5,4,-0.5


Databricks visualization. Run in Databricks to view.

a turbina 5 apresenta subperformance a partir de 10 m/s até 18 m/s em relação às outras turbinas. após 18 m/s é esperado queda de performance pelo manual e também é sinal de dados nulos (o que justifica o serrilhado final).

vou averiguar se realmente a turbina 1 apresenta uma subperformance ou se é um sintoma de pouco volume de dados válidos para aquela faixa de ventos.

In [0]:
%sql
SELECT
    turbina_id,
    round(avg(power_kw),1) AS potencia_media_kw,
    count(*) AS n_leituras
FROM workspace.default.gold_fato_scada_curado
WHERE power_kw IS NOT NULL
AND wind_speed_m_s BETWEEN 13 AND 18 --faixa alta
GROUP BY turbina_id
ORDER BY potencia_media_kw;

turbina_id,potencia_media_kw,n_leituras
5,1674.7,569
1,1924.0,711
6,1976.9,389
3,1981.4,638
4,2005.9,743
2,2006.0,735


### resposta da pergunta:
A curva de potência agregada reproduz a curva oficial do fabricante (cut-in ~3 m/s, saturação ~2050 kW a partir de ~12,5 m/s), validando o pipeline de ponta a ponta.

A análise por turbina revelou subperformance na turbina 5: na faixa de vento forte (13-18 m/s), ela entrega potência média de 1674,7 kW contra ~2006 kW da referência (-16%). A amostra da turbina 5 é robusta e comparável à das demais (569 leituras válidas, contra 638-743 das saudáveis), o que descarta escassez de dados como causa e confirma a subperformance como característica real da máquina. A hipótese inicial de que a altura de cubo reduzida (turbinas 3 e 6) causaria subperformance não se confirmou pois essas turbinas operam normalmente. A causa é, portanto, específica da turbina 5, sugerindo investigação de manutenção.

# 2. Disponibilidade
qual a fração de tempo em cada status (normal, standby, aviso, parada) por turbina? Quais perdem mais tempo produtivo?

In [0]:
%sql
SELECT
    turbina_id,
    count(*) AS total_intervalos,
    sum(CASE WHEN status_id = 1 THEN 1 ELSE 0 END) AS intervalos_parada,
    round(100.0 * sum(CASE WHEN status_id = 1 THEN 1 ELSE 0 END)/count(*),2) AS pct_parada,
    round(100.0 * sum(CASE WHEN status_id = 0 THEN 1 ELSE 0 END)/count(*),2) AS pct_operacao
FROM workspace.default.gold_fato_scada_curado
GROUP BY turbina_id
ORDER BY pct_operacao DESC

turbina_id,total_intervalos,intervalos_parada,pct_parada,pct_operacao
6,52560,2097,3.99,96.01
4,52560,2105,4.00,96.00
2,52560,2164,4.12,95.88
3,52560,2283,4.34,95.66
5,52560,2312,4.40,95.60
1,52560,2328,4.43,95.57


Databricks visualization. Run in Databricks to view.

### resposta da pergunta:

Todas as seis turbinas apresentam frações de parada (status Stop) semelhantes, entre ~3,7% e ~4,3% do tempo, sem nenhuma turbina destoando significativamente em indisponibilidade. Notavelmente, a turbina 5 identificada como subperformante na Pergunta 1 não está entre as que mais param. Isso refina o diagnóstico da pergunta 1: a menor geração da turbina 5 decorre de perda de eficiência durante a operação, e não de maior tempo parada. A causa provável situa-se em desgaste/calibração (pás, pitch ou gerador), e não em disponibilidade.

# 3. Temperatura × operação
 como as temperaturas (mancal, gerador, multiplicadora) evoluem com carga/vento? Aparece temperatura anormal antes de eventos críticos?

não vou comparar em condições de operação diferentes, afinal uma em operação é quente e parada é fria. a ideia é comparar na mesma condição de carga em operação.

In [0]:
%sql
SELECT
    turbina_id,
    count(*) AS n_leituras,
    round(avg(front_bearing_temp_c),1) AS mancal_diant,
    round(avg(rear_bearing_temp_c),1) AS mancal_tras,
    round(avg(gear_oil_temp_c),1) AS oleo_mult,
    round(avg(generator_bearing_front_temp_c),1) AS mancal_gerador,
    round(avg(nacelle_ambient_temp_c),1) AS ambiente
FROM workspace.default.gold_fato_scada_curado
WHERE power_kw > 1500 --apenas em carga alta, plena operacao
GROUP BY turbina_id
ORDER BY turbina_id;


turbina_id,n_leituras,mancal_diant,mancal_tras,oleo_mult,mancal_gerador,ambiente
1,6689,74.3,68.3,51.8,31.3,9.2
2,7854,72.4,69.6,52.0,32.6,8.9
3,6244,74.4,68.5,51.6,32.0,8.2
4,6961,73.2,69.0,50.6,31.3,9.0
5,6053,73.0,67.8,50.5,30.3,8.2
6,4980,75.3,69.4,51.9,31.8,9.5


interessante:  a turnina com menor performance em alta (turbina 5) é a que possui temperaturas mais frias de todas nos componentes mecanicos. isso indica que não tem perda de eficiencia por atrito, desgaste mecanico ou superaquecimento. provavelmente é má disposição aerodinâmica.

vou fazer uma nova query considerando o angulo das pás em relação dentro da janela de vento forte (entre 13 e 18 m/s), que é onde ela tem subperformance.

In [0]:
%sql
SELECT
  turbina_id,
  count(*)                            AS n_leituras,
  round(avg(blade_pitch_a_deg), 2)    AS pitch_medio,
  round(avg(power_kw), 1)             AS potencia_media,
  round(avg(rotor_speed_rpm), 2)      AS rotor_medio
FROM workspace.default.gold_fato_scada_curado
WHERE wind_speed_m_s BETWEEN 13 AND 18   -- faixa onde a turbina 5 derrapa
  AND power_kw IS NOT NULL
GROUP BY turbina_id
ORDER BY turbina_id;

turbina_id,n_leituras,pitch_medio,potencia_media,rotor_medio
1,711,15.83,1924.0,14.31
2,735,12.77,2006.0,14.92
3,638,13.78,1981.4,14.73
4,743,12.36,2005.9,14.91
5,569,24.86,1674.7,12.52
6,389,13.42,1976.9,14.72


Databricks visualization. Run in Databricks to view.

### resposta:

Na faixa de vento forte (13-18 m/s), identificou-se a anomalia: o ângulo de pitch médio da turbina 5 é 24,86°, contra 12-14° das turbinas saudáveis — praticamente o dobro. O pitch excessivo fecha as pás além do necessário, reduzindo a rotação (12,52 vs ~14,9 rpm) e a potência (1674,7 vs ~2006 kW). Conclui-se que a causa-raiz é o sistema de controle de pitch descalibrado, fechando as pás em excesso sob vento forte. Recomendação: inspeção e recalibração do controle de pitch da turbina 5 (e monitoramento da turbina 1, com pitch 15,83°, discretamente elevado).

# 4. Eventos no tempo
distribuição dos eventos do log por tipo e por período: quais categorias são mais frequentes e quais mais custam em tempo parado?

Um bom gestor de manutenção quer saber a frequência de eventos, mas também a natureza deles. Um evento raro mais caro pode ser pior do que muitos logs mais baratos, menos graves.

para isso vou puxar a camada prata (silver) que possui o log de eventos.

In [0]:
%sql
SELECT
    status,
    qtd_eventos,
    --count(*) AS qtd_eventos,
    round(100.0 * qtd_eventos / sum(qtd_eventos) OVER (), 1) AS pct_do_total
FROM (
    SELECT
        status,
        count(*) AS qtd_eventos
    FROM workspace.default.silver_status
    GROUP BY status
)
ORDER BY qtd_eventos DESC;

status,qtd_eventos,pct_do_total
Informational,68608,98.1
Stop,733,1.0
Warning,524,0.7
Communication,98,0.1


In [0]:
%sql
SELECT
   status,
   count(*) AS qtd_eventos,
   round(sum((unix_timestamp(tempo_fim)-unix_timestamp(tempo_inicio))/3600.0),1) AS horas_totais,
   round(avg((unix_timestamp(tempo_fim)-unix_timestamp(tempo_inicio))/60.0),1) AS dur_media_minuto
FROM workspace.default.silver_status
WHERE tempo_fim IS NOT NULL
GROUP BY status
ORDER BY horas_totais DESC;

status,qtd_eventos,horas_totais,dur_media_minuto
Warning,524,6776.4,775.9
Stop,733,2207.0,180.7
Communication,98,1862.1,1140.1
Informational,8307,431.7,3.1


Databricks visualization. Run in Databricks to view.

### resposta:

A análise separou frequência de custo. Em frequência, eventos Informational dominam (98,1%), mas são instantâneos (duração média 3,1 min) e irrelevantes em tempo perdido. Em tempo total de indisponibilidade, o ranking se inverte: Warning lidera com 6.776 h (média de 13 h por evento), seguido de Stop com 2.207 h (3 h/evento) e Communication com 1.862 h, este raro (98 eventos) mas com a maior duração média (19 h). Conclusão gerencial: o volume de eventos não indica onde está o tempo perdido; embora Stop seja mais frequente, os eventos Warning custam 3× mais tempo produtivo. Isso indica que a atenção de manutenção deveria priorizar a investigação dos Warning, e reforça a recomendação de expandir a reconciliação (hoje restrita a Stop) para incluir Warnings

## 5. Qualidade e governança na distribuição
o dado está distribuído uniformemente ao longo do ano, ou tem buracos/meses falhos?

In [0]:
%sql
SELECT
  turbina_id,
  month(tempo_id)                          AS mes,
  count(*)                                 AS n_leituras,
  round(100.0 * count(power_kw) / count(*), 1)  AS pct_power_preenchido
FROM workspace.default.gold_fato_scada_curado
GROUP BY turbina_id, month(tempo_id)
ORDER BY turbina_id, mes;

turbina_id,mes,n_leituras,pct_power_preenchido
1,1,4464,99.7
1,2,4032,100.0
1,3,4464,99.8
1,4,4320,100.0
1,5,4464,99.6
1,6,4320,96.8
1,7,4464,99.9
1,8,4464,100.0
1,9,4320,100.0
1,10,4464,69.4


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT
  turbina_id,
  round(100.0 * count(power_kw) / count(*), 1)  AS pct_preenchido_out
FROM workspace.default.gold_fato_scada_curado
WHERE month(tempo_id) = 10
GROUP BY turbina_id
ORDER BY turbina_id;

turbina_id,pct_preenchido_out
1,69.4
2,69.4
3,69.4
4,69.3
5,69.4
6,69.4


A qualidade dos dados foi avaliada em três pilares:

Completude: ~3,5% de valores nulos, homogêneos entre colunas --> reflexo físico das paradas (turbina parada não reporta nenhum sensor), não de falha de coleta.

Validade: a auditoria de domínio, com faixas ancoradas no manual do fabricante (Senvion MM92), identificou 2.031 leituras de rotação de gerador negativas (0,6%) — fisicamente impossíveis, coincidentes 100% com potência nula e rotor parado, caracterizadas como ruído de sensor e tratadas como inválidas na camada curada. O ângulo de pitch, inicialmente suspeito, foi validado como correto após identificar sua convenção circular (0–360°).

Cobertura temporal: a análise mês a mês por turbina revelou uma queda acentuada e uniforme em outubro/2018 — todas as seis turbinas com ~69,4% de preenchimento, contra ~96% nos demais meses. A uniformidade entre turbinas independentes indica um evento sistêmico (falha no sistema central de coleta ou parada geral do parque), e não falhas individuais. Recomenda-se investigar os registros operacionais de outubro/2018.

Conclusão: excetuando o evento sistêmico de outubro e o ruído pontual do sensor de gerador (ambos identificados e documentados), a base apresenta alta qualidade e é confiável para as análises propostas.